<a href="https://colab.research.google.com/github/j156734119/Amazon-reviews-2023-electronics-SFT-DPO/blob/main/Amazon_Review_Alignment_A100%E2%80%941.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amazon Review Alignment: A100 Formal Pipeline

使用 `Qwen/Qwen3.5-2B`、BF16 与 4-bit QLoRA 执行正式的五模型实验。
该配置控制了 PPO/GRPO prompt 数与评估规模，但 Colab Compute Unit
消耗是动态的，不能保证固定在 100 CU 内。


## 1. 挂载 Drive 并加载项目

本 Notebook 将仓库放在 Google Drive，训练输出和 checkpoint 会在断开
Colab 后保留。目标 GPU：`A100`。

开始前必须先将本地最新代码和本 Notebook 提交并推送到 GitHub `main`
分支，否则下方 `git clone` 会获取旧版本。


In [11]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/j156734119/Amazon-reviews-2023-electronics-SFT-DPO.git"
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--short"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if status:
        print("Preserving Colab-local tracked changes before pull:")
        print(status)
        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "stash",
                "push",
                "-m",
                "colab-auto-stash-before-pull",
            ],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Preserving Colab-local tracked changes before pull:
?? a100_mini_results.zip
Repository: /content/drive/MyDrive/amazon-review-alignment-workspace/repo


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

## 2. 安装依赖

执行后使用 Colab 菜单 **运行时 -> 重新启动会话**。重启后从下一单元格
继续，不需要再次执行安装。


In [12]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
    check=True,
)
print("Installation complete. Restart the Colab runtime now.")


Installation complete. Restart the Colab runtime now.


## 3. 重启后恢复目录、加载 Secrets

在 Colab 左侧钥匙图标中添加 `OPENAI_API_KEY`。`HF_TOKEN` 对公开模型
是可选的，但能提高 Hugging Face 下载限额。


In [13]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
os.chdir(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
CONFIG = "configs/rlhf_a100.yaml"

try:
    import amazon_review_alignment
    import bitsandbytes
    import peft
    import transformers
    import trl
except (ImportError, ModuleNotFoundError):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".[train,eval,dev]",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    import amazon_review_alignment
    import bitsandbytes
    import peft
    import transformers
    import trl

for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[secret_name] = value

def cli(*arguments: str, check: bool = True) -> subprocess.CompletedProcess:
    command = [
        sys.executable,
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]
    print("\n$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    returncode = process.wait()
    result = subprocess.CompletedProcess(
        command,
        returncode,
        stdout="".join(output_lines),
        stderr=None,
    )
    if check and returncode:
        raise RuntimeError(
            f"Command failed with exit code {returncode}: "
            + " ".join(command)
        )
    return result

print("Config:", CONFIG)
print("Package:", Path(amazon_review_alignment.__file__).resolve())
print(
    "Training stack:",
    transformers.__version__,
    trl.__version__,
    peft.__version__,
    bitsandbytes.__version__,
)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config: configs/rlhf_a100.yaml
Package: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/src/amazon_review_alignment/__init__.py
Training stack: 5.5.4 1.5.1 0.19.1 0.49.2
OpenAI key loaded: True
HF token loaded: False


## A100 环境检查


In [14]:
import importlib.metadata

import torch
import transformers
import trl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a Colab GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {total_gib:.2f} GiB")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", importlib.metadata.version("peft"))

if "A100".lower() not in gpu_name.lower():
    raise RuntimeError("Expected A100, but Colab assigned: " + gpu_name)
if total_gib < 38:
    raise RuntimeError("Insufficient GPU memory for this profile.")
if True and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This profile requires BF16 support.")


GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GiB
PyTorch: 2.11.0+cu128
Transformers: 5.5.4
TRL: 1.5.1
PEFT: 0.19.1


## 4. 选择 A100 Mini Smoke 或正式训练

首次运行保持 `RUN_MODE="mini"`。它使用 Qwen3.5-2B 和真实 A100
装配，但只训练一步，并使用 60 条评论验证完整链路。全部通过后改成
`RUN_MODE="formal"`，重新从数据准备开始执行正式实验。

Mini 与正式输出目录完全隔离，不会相互复用 checkpoint。


In [15]:
import yaml

from amazon_review_alignment.config import load_config

RUN_MODE = "formal"  # mini | formal

if RUN_MODE == "mini":
    merged = load_config(REPO_DIR / "configs" / "rlhf_a100.yaml")
    merged.pop("_config_path", None)

    old_root = "outputs/a100-qwen3.5-2b"
    new_root = "outputs/a100-mini-qwen3.5-2b"

    def replace_output_paths(value):
        if isinstance(value, dict):
            return {
                key: replace_output_paths(item)
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [replace_output_paths(item) for item in value]
        if isinstance(value, str):
            return value.replace(old_root, new_root)
        return value

    merged = replace_output_paths(merged)
    merged["project"]["output_dir"] = new_root
    merged["data"].update(
        {
            "sample_size": 60,
            "max_scanned_reviews": 20000,
            "rating_targets": {
                "1": 12,
                "2": 12,
                "3": 12,
                "4": 12,
                "5": 12,
            },
            "splits": {
                "train": 42,
                "validation": 6,
                "test": 12,
            },
        }
    )
    merged["teacher"].update(
        {
            "pilot_size": 5,
            "max_estimated_cost_usd": 1.0,
        }
    )
    merged["training"]["sft"]["max_steps"] = 1
    merged["training"]["dpo"]["max_steps"] = 1
    merged["rlhf"].update(
        {
            "human_calibration_samples": 0,
            "ai_reward_train_pairs": 4,
            "ai_reward_validation_pairs": 2,
            "ppo_prompt_count": 4,
        }
    )
    merged["rlhf"]["reward"]["max_steps"] = 1
    merged["rlhf"]["ppo"]["total_episodes"] = 4
    merged["rlhf"]["ppo"]["gradient_accumulation_steps"] = 1
    merged["rlhf"]["ppo"]["save_steps"] = 1
    merged["rlhf"]["grpo"]["prompt_count"] = 4
    merged["rlhf"]["grpo"]["max_steps"] = 1
    merged["evaluation"]["max_test_samples"] = 4

    mini_path = Path("/content/rlhf_a100_mini.yaml")
    mini_path.write_text(
        yaml.safe_dump(merged, sort_keys=False),
        encoding="utf-8",
    )
    CONFIG = str(mini_path)
    effective = merged
elif RUN_MODE == "formal":
    CONFIG = "configs/rlhf_a100.yaml"
    effective = load_config(REPO_DIR / CONFIG)
else:
    raise ValueError("RUN_MODE must be 'mini' or 'formal'.")

print("Run mode:", RUN_MODE)
print("Effective config:", CONFIG)
print(
    "PPO auxiliary models in 4-bit:",
    effective["rlhf"]["ppo"]["auxiliary_model_load_in_4bit"],
)


Run mode: formal
Effective config: configs/rlhf_a100.yaml
PPO auxiliary models in 4-bit: False


## 4. 测试、准备数据并生成 Base baseline


In [16]:
subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=REPO_DIR,
    check=True,
)
cli("prepare-data", "--config", CONFIG)



$ /usr/bin/python3 -m amazon_review_alignment.cli prepare-data --config configs/rlhf_a100.yaml
2026-06-15 03:08:09,300 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-06-15 03:08:09,753 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-06-15 03:08:09,754 | INFO | datasets | JAX version 0.7.2 available.
2026-06-15 03:08:10,272 | INFO | amazon_review_alignment.data | Streaming reviews from https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl
2026-06-15 03:08:10,438 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
2026-06-15 03:08:10,593 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/revision/main "HTTP/1.1 200 OK"
2026-06-15 03:08:10,645 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw%2Frev

CompletedProcess(args=['/usr/bin/python3', '-m', 'amazon_review_alignment.cli', 'prepare-data', '--config', 'configs/rlhf_a100.yaml'], returncode=0, stdout='2026-06-15 03:08:09,300 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.\n2026-06-15 03:08:09,753 | INFO | datasets | TensorFlow version 2.20.0 available.\n2026-06-15 03:08:09,754 | INFO | datasets | JAX version 0.7.2 available.\n2026-06-15 03:08:10,272 | INFO | amazon_review_alignment.data | Streaming reviews from https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl\n2026-06-15 03:08:10,438 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"\n2026-06-15 03:08:10,593 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/revision/main "HTTP/1.1 200 OK"\n2026-06-15 03:08:10,645 | INFO | httpx | HTTP Request: GET https://huggingface.

In [17]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "--force-inference",
)

import pandas as pd
from amazon_review_alignment.config import load_config

output_root = Path(
    load_config(CONFIG)["project"]["output_dir"]
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))



$ /usr/bin/python3 -m amazon_review_alignment.cli evaluate --config configs/rlhf_a100.yaml --variants base --force-inference
2026-06-15 03:08:20,328 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-06-15 03:08:27,842 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 03:08:27,843 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-15 03:08:27,872 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-06-15 03:08:27,904 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-06-15 03:08:27,976 | INFO | httpx |

KeyboardInterrupt: 

## 5. 教师数据

Pilot 会立即调用 OpenAI API。Batch 提交后可能需要等待；提交成功后可以
关闭 GPU Runtime，稍后重新连接并重复“检查 Batch”单元格。


In [18]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
cli("teacher-pilot", "--config", CONFIG)



$ /usr/bin/python3 -m amazon_review_alignment.cli teacher-pilot --config configs/rlhf_a100.yaml
2026-06-15 03:10:49,309 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-15 03:10:51,105 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-15 03:10:54,385 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-15 03:10:56,843 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-15 03:10:58,903 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-15 03:11:01,708 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-15 03:11:04,997 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-15 03:11:11,006 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1

KeyboardInterrupt: 

In [19]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive, userdata

drive.mount("/content/drive")

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
CONFIG = "configs/rlhf_a100.yaml"

# 进入项目目录，保证相对配置路径有效
os.chdir(REPO_DIR)

# 加载 Colab Secrets
for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None

    if value:
        os.environ[secret_name] = value

# 重新安装当前项目源码
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

def cli(*arguments, check=True):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]

    print("\n$", " ".join(command), flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    return_code = process.wait()

    if check and return_code:
        print("\n===== LAST 200 LINES =====")
        print("".join(lines[-200:]))
        raise RuntimeError(
            f"Command failed with exit code {return_code}"
        )

    return subprocess.CompletedProcess(
        command,
        return_code,
        stdout="".join(lines),
    )

print("Repository:", REPO_DIR)
print("Config:", CONFIG)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository: /content/drive/MyDrive/amazon-review-alignment-workspace/repo
Config: configs/rlhf_a100.yaml
OpenAI key loaded: True
HF token loaded: False


In [20]:
# 第一次执行会提交 Batch；后续重复执行会查询并下载结果。
cli("teacher-batch", "--config", CONFIG)



$ /usr/bin/python3 -u -m amazon_review_alignment.cli teacher-batch --config configs/rlhf_a100.yaml
2026-06-15 03:12:19,734 | INFO | httpx | HTTP Request: GET https://api.openai.com/v1/batches/batch_6a2f615a5d4c81908c06a49ef4bae222 "HTTP/1.1 200 OK"
2026-06-15 03:12:21,354 | INFO | httpx | HTTP Request: GET https://api.openai.com/v1/files/file-YSRyMTNErQUrNJ3LpVLVpA/content "HTTP/1.1 200 OK"
{
  "completed": {
    "train": 1033,
    "validation": 130
  },
  "quarantined": 2837,
  "usage": {
    "input_tokens": 1786035,
    "output_tokens": 555984
  },
  "batch_cost_usd": 1.920727
}


CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'amazon_review_alignment.cli', 'teacher-batch', '--config', 'configs/rlhf_a100.yaml'], returncode=0, stdout='2026-06-15 03:12:19,734 | INFO | httpx | HTTP Request: GET https://api.openai.com/v1/batches/batch_6a2f615a5d4c81908c06a49ef4bae222 "HTTP/1.1 200 OK"\n2026-06-15 03:12:21,354 | INFO | httpx | HTTP Request: GET https://api.openai.com/v1/files/file-YSRyMTNErQUrNJ3LpVLVpA/content "HTTP/1.1 200 OK"\n{\n  "completed": {\n    "train": 1033,\n    "validation": 130\n  },\n  "quarantined": 2837,\n  "usage": {\n    "input_tokens": 1786035,\n    "output_tokens": 555984\n  },\n  "batch_cost_usd": 1.920727\n}\n')

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
CONFIG = str(REPO_DIR / "configs" / "rlhf_a100.yaml")

if not REPO_DIR.exists():
    raise RuntimeError(f"项目目录不存在：{REPO_DIR}")

# 加载 Secrets
for name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None

    if value:
        os.environ[name] = value

# 重新安装本地项目
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

# 确保当前 Python 进程能立即找到 src package
src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.chdir(REPO_DIR)

from amazon_review_alignment.config import load_config

def cli(*arguments, check=True):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(arg) for arg in arguments),
    ]

    print("\n$", " ".join(command), flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    return_code = process.wait()

    if check and return_code:
        print("".join(lines[-200:]))
        raise RuntimeError(
            f"Command failed with exit code {return_code}"
        )

    return subprocess.CompletedProcess(
        command,
        return_code,
        stdout="".join(lines),
    )

print("Repository:", REPO_DIR)
print("Config:", CONFIG)
print("Package loaded successfully")
print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")))

In [21]:
from pathlib import Path

from amazon_review_alignment.config import load_config

effective_config = load_config(CONFIG)
output_root = (
    REPO_DIR / effective_config["project"]["output_dir"]
).resolve()

train_preferences = (
    output_root / "teacher" / "preferences_train.jsonl"
)
validation_preferences = (
    output_root / "teacher" / "preferences_validation.jsonl"
)

print("Output root:", output_root)
print("Train file exists:", train_preferences.exists())
print(
    "Validation file exists:",
    validation_preferences.exists(),
)

# 文件不存在时，查询已有 Batch 并尝试下载结果
if not train_preferences.exists() or not validation_preferences.exists():
    cli("teacher-batch", "--config", CONFIG)

if not train_preferences.exists() or not validation_preferences.exists():
    raise RuntimeError(
        "OpenAI Batch 尚未完成。可以断开 A100，稍后重新运行本单元格。"
    )

train_rows = sum(
    1 for line in train_preferences.open(encoding="utf-8")
    if line.strip()
)
validation_rows = sum(
    1 for line in validation_preferences.open(encoding="utf-8")
    if line.strip()
)

print("Teacher train rows:", train_rows)
print("Teacher validation rows:", validation_rows)

if train_rows == 0 or validation_rows == 0:
    raise RuntimeError("Teacher 数据文件存在，但没有有效记录。")

print("Teacher data ready. 可以开始 SFT。")

Output root: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b
Train file exists: True
Validation file exists: True
Teacher train rows: 1033
Teacher validation rows: 130
Teacher data ready. 可以开始 SFT。


## 6. SFT、合并权重与 DPO


In [22]:
cli("train-sft", "--config", CONFIG)



$ /usr/bin/python3 -u -m amazon_review_alignment.cli train-sft --config configs/rlhf_a100.yaml
2026-06-15 03:12:47,215 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-06-15 03:12:47,674 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-06-15 03:12:47,675 | INFO | datasets | JAX version 0.7.2 available.
2026-06-15 03:12:57,906 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 03:12:57,907 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-15 03:12:57,922 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-06-15 03:12:57,995 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/mai

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'amazon_review_alignment.cli', 'train-sft', '--config', 'configs/rlhf_a100.yaml'], returncode=0, stdout='2026-06-15 03:12:47,215 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.\n2026-06-15 03:12:47,674 | INFO | datasets | TensorFlow version 2.20.0 available.\n2026-06-15 03:12:47,675 | INFO | datasets | JAX version 0.7.2 available.\n2026-06-15 03:12:57,906 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"\nWarning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n2026-06-15 03:12:57,907 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n2026-06-15 03:12:57,922 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-ca

In [23]:
cli("merge-sft", "--config", CONFIG)



$ /usr/bin/python3 -u -m amazon_review_alignment.cli merge-sft --config configs/rlhf_a100.yaml
2026-06-15 03:22:13,264 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-06-15 03:22:14,560 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 03:22:14,561 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-15 03:22:14,611 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-06-15 03:22:14,812 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 03:22:14,872 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/model

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'amazon_review_alignment.cli', 'merge-sft', '--config', 'configs/rlhf_a100.yaml'], returncode=0, stdout='2026-06-15 03:22:13,264 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.\n2026-06-15 03:22:14,560 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"\nWarning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n2026-06-15 03:22:14,561 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n2026-06-15 03:22:14,611 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"\n2026-06-15 03:22:14,812 | INFO | httpx | HTTP Request: HE

In [24]:
cli("train-dpo", "--config", CONFIG)



$ /usr/bin/python3 -u -m amazon_review_alignment.cli train-dpo --config configs/rlhf_a100.yaml
2026-06-15 03:22:37,010 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-06-15 03:22:37,524 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-06-15 03:22:37,526 | INFO | datasets | JAX version 0.7.2 available.
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 1/320 [00:00<02:59,  1.78it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 320/320 [00:02<00:00, 133.91it/s]
warmup_ratio is deprecated and

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'amazon_review_alignment.cli', 'train-dpo', '--config', 'configs/rlhf_a100.yaml'], returncode=0, stdout='2026-06-15 03:22:37,010 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.\n2026-06-15 03:22:37,524 | INFO | datasets | TensorFlow version 2.20.0 available.\n2026-06-15 03:22:37,526 | INFO | datasets | JAX version 0.7.2 available.\nThe fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d\n\nLoading weights:   0%|          | 0/320 [00:00<?, ?it/s]\nLoading weights:   0%|          | 1/320 [00:00<02:59,  1.78it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.\n  tor

In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "sft",
    "dpo",
    "--force-inference",
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))



$ /usr/bin/python3 -u -m amazon_review_alignment.cli evaluate --config configs/rlhf_a100.yaml --variants base sft dpo --force-inference
2026-06-15 03:43:47,740 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-06-15 03:43:55,000 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 03:43:55,000 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-15 03:43:55,019 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-06-15 03:43:55,093 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 03:43:55,111 | INFO | httpx | HTTP Request: 

## 7. 构建纯 RLAIF 数据

A100 正式流程不要求人工填写 200 条 A/B。Reward Model、PPO 和 GRPO
直接使用 OpenAI 教师生成并通过规则校验的 chosen/rejected 偏好。

这属于 RLAIF，而不是纯 RLHF。独立的 200 条人工盲评仅用于最终评估，
不进入训练数据。


In [ ]:
cli("build-rlhf-data", "--config", CONFIG)

import json

manifest_path = output_root / "rlhf" / "data_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))
assert manifest["alignment_method"] == "rlaif"
assert manifest["human_total_rows"] == 0


## 8. Reward Model、PPO 与 GRPO

每个阶段是独立单元格。阶段失败时先处理报错，不要跳过并继续。


In [ ]:
cli("train-reward", "--config", CONFIG)


In [ ]:
cli("train-ppo", "--config", CONFIG)


In [ ]:
cli("train-grpo", "--config", CONFIG)


## 9. 五模型统一评估和报告


In [ ]:
cli(
    "evaluate",
    "--config",
    str(CONFIG),
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
)

cli("build-report", "--config", str(CONFIG))

metrics_path = output_root / "evaluation" / "metrics.csv"
report_path = output_root / "report.md"

display(pd.read_csv(metrics_path))
print("Report:", report_path.resolve())